# Raw Crude Oil Intraday Data Exploration

This notebook documents the first inspection of the raw Databento 1-minute data before cleaning and aggregation.

The source is `data/raw/crude_oil_conflict_1m.dbn`, downloaded from `GLBX.MDP3` using the `CL.FUT` parent symbol. The query covers January 1, 2025 through October 7, 2026 so that the pre-conflict baseline and the current conflict period can be compared. The raw file remains unchanged; filtering, session rules, and 15-minute aggregation belong in the preparation workflow.

In [3]:
from pathlib import Path

import databento as db
import pandas as pd


candidates = [
    Path("data/raw/crude_oil_conflict_1m.dbn"),
    Path("../data/raw/crude_oil_conflict_1m.dbn"),
]
raw_path = next(path for path in candidates if path.exists())
intraday_store = db.DBNStore.from_file(raw_path)
intraday = intraday_store.to_df()
intraday.index.name = "ts_event"

print(f"Loaded {raw_path}")
print(f"Rows: {len(intraday):,}")
intraday.head()

Loaded ../data/raw/crude_oil_conflict_1m.dbn
Rows: 15,196,660


,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
ts_event,,,,,,,,,
2025-01-01 23:00:00+00:00,33,1,279702,0.68,0.68,0.68,0.68,1,CLU5-CLX5
2025-01-01 23:00:00+00:00,33,1,195761,0.88,0.88,0.88,0.88,2,CLG5-CLJ5
2025-01-01 23:00:00+00:00,33,1,335115,1.26,1.27,1.26,1.27,4,CLG5-CLK5
2025-01-01 23:00:00+00:00,33,1,275422,0.36,0.36,0.36,0.36,1,CLJ5-CLK5
2025-01-01 23:00:00+00:00,33,1,625061,70.97,70.97,70.94,70.94,2,CLJ5


## Intraday coverage and schema

The query returns 1-minute OHLCV records. The requested 15-minute bars are not downloaded directly; they should be aggregated from these 1-minute observations after instrument and session rules are finalized.

In [4]:
coverage = pd.Series(
    {
        "rows": len(intraday),
        "columns": len(intraday.columns),
        "unique_symbols": intraday["symbol"].nunique(),
        "unique_instrument_ids": intraday["instrument_id"].nunique(),
        "first_timestamp": intraday.index.min(),
        "last_timestamp": intraday.index.max(),
    }
)
coverage

rows                                      15196660
columns                                          9
unique_symbols                                1128
unique_instrument_ids                         1128
first_timestamp          2025-01-01 23:00:00+00:00
last_timestamp           2026-10-07 23:59:00+00:00
dtype: object

## Intraday quality checks

These checks describe the records returned by Databento before any session filtering, contract filtering, or resampling.

In [6]:
ohlcv_columns = ["open", "high", "low", "close", "volume"]
quality = pd.Series(
    {
        "missing_ohlcv_values": int(intraday[ohlcv_columns].isna().sum().sum()),
        "missing_symbols": int(intraday["symbol"].isna().sum()),
        "duplicate_timestamp_instrument_rows": int(
            intraday.reset_index().duplicated(subset=["ts_event", "instrument_id"]).sum()
        ),
        "zero_volume_rows": int((intraday["volume"] == 0).sum()),
        "minimum_low": intraday["low"].min(),
        "maximum_high": intraday["high"].max(),
    }
)
quality

missing_ohlcv_values                     0.00
missing_symbols                          0.00
duplicate_timestamp_instrument_rows      0.00
zero_volume_rows                         0.00
minimum_low                            -20.69
maximum_high                           119.48
dtype: float64

## Instrument mix and session coverage

The parent query includes outright contracts, spreads, and related instruments. The labels below are exploratory only; final filtering should use instrument metadata and retain `instrument_id`.

In [7]:
intraday["instrument_class"] = "other"
intraday.loc[
    intraday["symbol"].str.match(r"^CL[A-Z][0-9]+$"),
    "instrument_class",
] = "outright_candidate"
intraday.loc[
    intraday["symbol"].str.contains("-", regex=False),
    "instrument_class",
] = "spread_candidate"

instrument_mix = intraday.groupby("instrument_class").agg(
    rows=("instrument_id", "size"),
    symbols=("symbol", "nunique"),
    instruments=("instrument_id", "nunique"),
)
instrument_mix

,rows,symbols,instruments
instrument_class,,,
other,11,6,6
outright_candidate,3641915,71,71
spread_candidate,11554734,1051,1051


In [8]:
session_coverage = (
    intraday.assign(
        date=intraday.index.date,
        utc_hour=intraday.index.hour,
    )
    .groupby("utc_hour")
    .agg(
        records=("instrument_id", "size"),
        instruments=("instrument_id", "nunique"),
    )
)
session_coverage

,records,instruments
utc_hour,,
0,335342,749
1,384726,755
2,297841,722
3,248554,712
4,207743,715
5,275672,746
6,411495,776
7,575134,817
8,703623,839


## 15-minute aggregation preview

The project proposes 15-minute bars, but the downloaded source is 1-minute data. This preview uses the outright candidate with the most 1-minute records and demonstrates the OHLCV aggregation rule. The production preparation code should apply the final timezone and session cutoff before resampling.

In [9]:
outright_intraday = intraday[
    intraday["instrument_class"].eq("outright_candidate")
].copy()

sample_instrument_id = outright_intraday.groupby("instrument_id").size().idxmax()
sample_symbol = outright_intraday.loc[
    outright_intraday["instrument_id"].eq(sample_instrument_id), "symbol"
].iloc[0]
sample = outright_intraday[
    outright_intraday["instrument_id"].eq(sample_instrument_id)
].sort_index()

bars_15m = sample[["open", "high", "low", "close", "volume"]].resample("15min").agg(
    {
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
    }
).dropna(subset=["open", "high", "low", "close"])

print(f"Sample instrument: {sample_symbol} | {sample_instrument_id}")
print(f"1-minute rows: {len(sample):,}")
print(f"15-minute bars: {len(bars_15m):,}")
bars_15m.head()

Sample instrument: CLZ6 | 447685
1-minute rows: 368,778
15-minute bars: 37,914


,open,high,low,close,volume
ts_event,,,,,
2025-01-02 00:15:00+00:00,65.97,65.97,65.97,65.97,2
2025-01-02 00:45:00+00:00,65.98,65.99,65.98,65.99,4
2025-01-02 01:00:00+00:00,65.98,66.06,65.98,65.98,18
2025-01-02 01:15:00+00:00,66.03,66.03,66.02,66.02,2
2025-01-02 01:30:00+00:00,66.08,66.08,66.05,66.05,33


## Preliminary findings

- The raw intraday file covers the pre-conflict baseline and the current conflict period.
- The parent-level query contains many instruments, so the raw file should not be treated as a single crude-oil price series.
- `instrument_id` must be retained because symbols can be reused across contract lifecycles.
- The final 15-minute dataset requires an explicit session timezone, closing cutoff, and treatment of incomplete bars.
- The reduced-quality dates reported by Databento should be recorded and reviewed before backtesting.